<a href="https://colab.research.google.com/github/rayj1981/2025-python-summer-prep/blob/main/Hwk4_4950.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Computation Graphs & Autograd**



In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import torch.optim as optim, matplotlib.pyplot as plt
torch.manual_seed(42)
print("PyTorch:", torch.__version__)

PyTorch: 2.10.0+cpu


In [ ]:
# ── Computation Graph ─────────────────────────────────────────────────
# PyTorch records every op into a graph. .backward() walks it in reverse,
# filling .grad on each leaf via the chain rule.

x = torch.tensor(2.0, requires_grad=True)
w = torch.tensor(3.0, requires_grad=True)
b = torch.tensor(1.0, requires_grad=True)
L = (x*w + b)**2          # dL/dx=42, dL/dw=28, dL/db=14
L.backward()
print("dL/dx:", x.grad, " dL/dw:", w.grad, " dL/db:", b.grad)

# ── Graph Control ─────────────────────────────────────────────────────
# .detach()         removes tensor from graph, SHARES storage
# .detach().clone() removes from graph, OWN storage (safe to mutate)
# model.train()     training mode
# model.eval()      inference mode (deterministic)
# torch.no_grad()   no graph built — faster, less memory

t = torch.tensor([1., 2., 3.], requires_grad=True)
y = t * 2
y.detach()[0] = 999
print("shared storage mutates original:", y.data)  # [999, 4, 6]

net = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 1))
opt = optim.Adam(net.parameters(), lr=1e-3)
Xtr, ytr = torch.randn(32, 4), torch.randn(32, 1)
Xvl, yvl = torch.randn(8, 4),  torch.randn(8, 1)

net.train(); opt.zero_grad()
loss = nn.MSELoss()(net(Xtr), ytr); loss.backward(); opt.step()
net.eval()
with torch.no_grad():
    vl = nn.MSELoss()(net(Xvl), yvl)
print(f"train={loss.item():.3f}  val={vl.item():.3f}  grad_fn={vl.grad_fn}")

# ── Exercise 2a: f(a,b) = tanh(a²+3b) at a=1, b=2
# Derive df/da and df/db analytically, then verify with autograd and finite diff (eps=1e-4)
# df/da = ???   df/db = ???
# YOUR CODE HERE

# ── Exercise 2b: fix the 3 bugs
m = nn.Sequential(nn.Linear(2, 4), nn.ReLU(), nn.Linear(4, 1))
o = optim.SGD(m.parameters(), lr=0.01)
Xb, yb = torch.randn(16, 2), torch.randn(16, 1)
o.zero_grad()                    # BUG 1
nn.MSELoss()(m(Xb), yb).backward(); o.step()
m.train()                        # BUG 2
preds = m(Xb)                    # BUG 3
print("grad_fn:", preds.clone().grad_fn)

dL/dx: tensor(42.)  dL/dw: tensor(28.)  dL/db: tensor(14.)
shared storage mutates original: tensor([999.,   4.,   6.])
train=1.387  val=0.474  grad_fn=None
grad_fn: <CloneBackward0 object at 0x78e965966590>


In [ ]:
# ── Custom Autograd ───────────────────────────────────────────────────
# Subclass torch.autograd.Function for custom forward + backward.
# ctx.save_for_backward() stashes tensors; return one grad per input.

class MyReLU(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x); return x.clamp(min=0)
    @staticmethod
    def backward(ctx, g):
        (x,) = ctx.saved_tensors; return g * (x > 0).float()

x1 = torch.randn(5, requires_grad=True)
x2 = x1.detach().clone().requires_grad_(True)
MyReLU.apply(x1).sum().backward(); F.relu(x2).sum().backward()
print("ReLU grads match:", torch.allclose(x1.grad, x2.grad))

# ── Exercise 3: implement MySigmoid  σ(x)=1/(1+e⁻ˣ),  dσ/dx=σ(1−σ)
# Store only the OUTPUT in ctx. Verify vs torch.sigmoid.
class MySigmoid(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        # YOUR CODE HERE
        pass
    @staticmethod
    def backward(ctx, grad_out):
        # YOUR CODE HERE
        pass
x = torch.randn(5, requires_grad=True)
# YOUR CODE HERE — check forward + gradients match torch.sigmoid

# ── Loss Functions ────────────────────────────────────────────────────
# MSELoss             → regression
# CrossEntropyLoss    → multi-class  (raw logits, NO softmax)
# BCEWithLogitsLoss   → binary       (raw logits, NO sigmoid)
# zero_grad() MUST come before backward() — grads accumulate by default

# ── Exercise 4a: run this, explain WHY w.grad grows (as a comment)
w = torch.tensor(1.0, requires_grad=True)
for i in range(3):
    (w*(i+1)).pow(2).backward()
    print(f"step {i+1}: w.grad={w.grad}")
# YOUR EXPLANATION HERE

# ── Exercise 4b: SGD + momentum from scratch  v←μv+g, θ←θ−lr·v  (μ=0.9, lr=0.05)
# Train nn.Linear(1,1) on (X, y_true) 200 epochs, plot vs optim.SGD(momentum=0.9)
torch.manual_seed(0)
X = torch.linspace(-1, 1, 100).unsqueeze(1)
y_true = 2*X + 1 + 0.1*torch.randn_like(X)
# YOUR CODE HERE

# ── Exercise 4c: MLP for binary classification (circles)
from sklearn.datasets import make_circles
Xn, yn = make_circles(n_samples=400, noise=0.1, factor=0.4, random_state=42)
Xall = torch.tensor(Xn, dtype=torch.float32)
yall = torch.tensor(yn, dtype=torch.float32).unsqueeze(1)
Xtr2, ytr2 = Xall[:300], yall[:300]; Xvl2, yvl2 = Xall[300:], yall[300:]
# Build a 2-layer MLP, train 100 epochs with BCEWithLogitsLoss
# Plot train loss / val loss / val accuracy
# YOUR CODE HERE

ReLU grads match: True
step 1: w.grad=2.0
step 2: w.grad=10.0
step 3: w.grad=28.0
